# 卷积神经网络 CNN

按代码单元格的原编号查阅课程手册。

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

torch.manual_seed(42)
torch.set_num_threads(min(torch.get_num_threads(), 4))
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
output_dir = Path(os.environ.get('COURSE_OUTPUT_DIR', 'outputs'))
output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv('15min_in.csv', usecols=['西直门'], encoding='gbk')
y = df['西直门'].to_numpy(dtype=float)
print(f'Device: {device}, series length: {len(y)}')
plt.figure(figsize=(12, 4))
plt.grid(True)
plt.plot(y)
plt.show()

In [ ]:
# 划分测试集和训练集，最后300个作为测试集
test_size = 300
train_iter = y[:-test_size]
test_iter = y[-test_size:]
# 归一化至[-1,1]区间，为了获得更好的训练效果
scaler = MinMaxScaler(feature_range=(-1, 1))
train_norm = scaler.fit_transform(train_iter.reshape(-1, 1))

# 创建时间序列训练集
train_set = torch.FloatTensor(train_norm).view(-1)

In [ ]:
# 定义时间窗口
Time_window_size = 72

# 从原时间序列中抽取出训练样本，用第1个值到第72个值作为X输入，预测第73个值作为y输出，这是一个用于训练的数据点，时间窗口向后滑动以此类推
def input_data(seq, ws):  
    out = []
    L = len(seq)
    for i in range(L-ws):
        window = seq[i:i + ws]
        label = seq[i + ws:i + ws + 1]
        out.append((window, label))
    return out


train_data = input_data(train_set, Time_window_size)
len(train_data)  # 等于1800（原始数据集长度）-300（测试集长度）-72（时间窗口）

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
output = sliding_window_view(train_set.numpy(), Time_window_size + 1)
print(f'Train windows: {len(output)}')

In [ ]:
class CNNnetwork(nn.Module):
    def __init__(self):
        super(CNNnetwork, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=2)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=32, kernel_size=2)
        self.pool = nn.MaxPool1d(kernel_size=2,stride=2)
        self.fc1 = nn.Linear(32*35, 640)
        self.fc2 = nn.Linear(640, 1)
        # self.fc1 = nn.Linear(32*70, 640)
        self.drop = nn.Dropout(0.5)
        
        
    def forward(self,x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.drop(x)
        x = x.flatten(start_dim=1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
    
net = CNNnetwork().to(device)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr= 0.0005)

In [ ]:
epochs = 20
net.train()
for epoch in range(epochs):
    total_loss = 0.
    for seq, y_train in train_data:
        optimizer.zero_grad()
        y_pred = net(seq.reshape(1, 1, -1).to(device))
        loss = criterion(y_pred, y_train.reshape(1, 1).to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.detach()
    print(f'Epoch: {epoch + 1:2} Mean loss: {total_loss.item() / len(train_data):.8f}')

In [ ]:
future = len(test_iter)
preds = train_set[-Time_window_size:].tolist()
net.eval()
with torch.no_grad():
    for _ in range(future):
        seq = torch.tensor(preds[-Time_window_size:], device=device).reshape(1, 1, -1)
        preds.append(net(seq).item())
true_predictions = scaler.inverse_transform(np.array(preds[Time_window_size:]).reshape(-1, 1)).ravel()
errors = true_predictions - test_iter
last_value = train_iter[-1]
print(f'Recursive test steps: {future}')
print(f'Recursive test MAE: {np.abs(errors).mean():.6f}')
print(f'Recursive test RMSE: {np.sqrt(np.mean(errors ** 2)):.6f}')
print(f'Fixed origin last observation RMSE: {np.sqrt(np.mean((last_value - test_iter) ** 2)):.6f}')

In [ ]:
plt.figure(figsize=(12, 4))
plt.grid(True)
plt.plot(y, label='Observed')
plt.plot(np.arange(len(train_iter), len(y)), true_predictions, label='Recursive forecast')
plt.legend()
plt.show()

In [ ]:
import torch 
import torch.nn as nn
from matplotlib import pyplot as plt
from PIL import Image
import torchvision
import torchvision.transforms as transforms
import numpy as np
import torch.utils.data as Data
import torch.optim as optim

# 首先对数据进行归一化处理，由于MINIST是一维的灰度图数据，所以mean和std只有一维
# Normalized_image = (image - mean) / std
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])])
train_dataset = torchvision.datasets.MNIST(root="./Datasets/MNIST", train=True, 
transform=transform, download=False)
test_dataset = torchvision.datasets.MNIST(root="./Datasets/MNIST", train=False, 
transform=transform, download=False)

batch_size = 64
train_loader = Data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = Data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
num_classes = 10
lr = 0.001
epochs = 20
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# pytorch封装卷积层
class ConvModule(nn.Module):
    def __init__(self):
        super(ConvModule, self).__init__()
        # 定义两层卷积层：
        self.conv2d = nn.Sequential(
            # 第一层 input_size = (1,28,28)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            nn.ReLU(inplace=True),  # inplace表示是否进行覆盖计算
            # 第二层
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            nn.ReLU(inplace=True),
        )
        # 输出层，将通道数变为分类数
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(64 * 7 * 7, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x):
        out = self.conv2d(x)
        # 将数据平整成一维
        out = out.view(-1, 64 * 7 * 7)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out


net = ConvModule().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(net.parameters(), lr=lr)

In [ ]:
def train_epoch(net, data_loader, device):
    net.train()
    total_loss = torch.zeros((), device=device)
    correct = torch.zeros((), device=device, dtype=torch.long)
    count = 0
    for inputs, labels in data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        output = net(inputs)
        loss = criterion(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        count += labels.numel()
        total_loss += loss.detach() * labels.numel()
        correct += (output.argmax(dim=1) == labels).sum()
    return total_loss.item() / count, correct.item() / count


def test_epoch(net, data_loader, device):
    net.eval()
    total_loss = torch.zeros((), device=device)
    correct = torch.zeros((), device=device, dtype=torch.long)
    count = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            output = net(inputs)
            loss = criterion(output, labels)
            count += labels.numel()
            total_loss += loss * labels.numel()
            correct += (output.argmax(dim=1) == labels).sum()
    return total_loss.item() / count, correct.item() / count

In [ ]:
# 存储每一个epoch的loss与acc的变化，便于后面的可视化
train_loss_list = []
train_acc_list = []
test_loss_list = []
test_acc_list = []

# 进行训练
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(net, data_loader=train_loader, device=device)
    test_loss, test_acc = test_epoch(net, data_loader=test_loader, device=device)

    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)
    test_loss_list.append(test_loss)
    test_acc_list.append(test_acc)
    print('epoch %d, train_loss %.6f, train_acc %.6f' % (epoch + 1, train_loss, train_acc))
    print('test_loss %.6f , test_acc %.6f' % (test_loss, test_acc)) 


In [ ]:
for name, train_values, test_values in [
    ('Loss', train_loss_list, test_loss_list),
    ('Acc', train_acc_list, test_acc_list),
]:
    fig, ax = plt.subplots()
    epochs_run = np.arange(1, len(train_values) + 1)
    ax.plot(epochs_run, train_values, label='train')
    ax.plot(epochs_run, test_values, label='test')
    ax.set(xlabel='Epoch', ylabel=name)
    ax.legend()
    fig.savefig(output_dir / f'{name}.jpg', dpi=150)
    plt.show()